# Greek vs English KV-cache cost

Thin wrapper around `experiments/plot_kv_cost.py`. Combines **T06 fertility** (`results/fertility.csv`) with the **T07 KV calculator** and real HF architecture configs to show how much more KV cache a Greek document costs vs the same English content, per candidate model.

The committed PNG lives in `results/figures/kv_greek_vs_english.png`.

## 1 — Load the measured fertility (from T06)

In [ ]:
from pathlib import Path

from experiments.plot_kv_cost import compute_rows, load_fertility, plot

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
tpw = load_fertility(ROOT / "results/fertility.csv")
tpw

## 2 — Compute KV bytes per (model, language)

`compute_rows` downloads each model's `config.json` (tiny, HF-cached), normalizes it via `arch_from_hf`, and applies the KV formula.

In [ ]:
N_WORDS = 1000
BITS = 16
rows = compute_rows(tpw, n_words=N_WORDS, bits=BITS)
for r in rows:
    print(f"{r.label:12s} {r.language}  n_tok={r.n_tokens:>6,}  KV={r.kv_mb:7.1f} MB")

## 3 — Plot and save

In [ ]:
out = ROOT / "results/figures/kv_greek_vs_english.png"
plot(rows, n_words=N_WORDS, bits=BITS, out_path=out)
print("wrote", out)

## 4 — Reading the figure

- **Krikri (Llama-arch, GQA 32:8):** ~**1.20×** KV for Greek. Same per-token cost as Meltemi; the gap is entirely tokenizer-driven.
- **Meltemi (Mistral-arch, GQA 32:8):** ~**1.04×** — near parity. Greek-adapted tokenizer cancels the Greek penalty.
- **Qwen2.5 (GQA 28:4):** the architecture is ~2.3× cheaper *per token* than Llama-arch, **but the Greek tokenizer inflates the token count by ~4.4×**, so the net Greek KV ends up the *largest* of any model here. The two forces don't cancel — they compound in the wrong direction.